# Generate SFT Dataset

## Load Library

In [1]:
import os
import re
import json
import base64
import requests
import time
import mimetypes
from tqdm import tqdm
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed

## Configuration

### Define Path

In [2]:
ROOT_DIR = Path("../data/metadata")
METADATA_PATH = ROOT_DIR / "mkn_house_metadata.json"

OUTPUT_DIR = Path("../data/sft_dataset")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_TRAIN_JSONL = OUTPUT_DIR / "train.jsonl"
OUTPUT_VAL_JSONL   = OUTPUT_DIR / "val.jsonl"
OUTPUT_TEST_JSONL  = OUTPUT_DIR / "test.jsonl"
OUTPUT_ALL_JSONL   = OUTPUT_DIR / "all.jsonl"

CACHE_PATH = OUTPUT_DIR / "openrouter_cache.json"

### Target Split

In [3]:
TARGET_SPLITS = ["train", "val", "test"]

### OpenRouter

In [4]:
API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL_NAME = "openai/gpt-5-mini"

### OpenRouter Config

In [5]:
TEMPERATURE = 0.1
MAX_TOKENS = 700
TIMEOUT = 90
MAX_RETRIES = 3
RETRY_SLEEP_SEC = 2
MAX_WORKERS = 3

### OpenRouter Prompt

#### Prompt CNN + VLM

In [6]:
# =========================================================
# SYSTEM PROMPT — CNN + VLM (SINGLE IMAGE)
# =========================================================

SYSTEM_PROMPT_CNNVLM_SINGLE = """
Anda adalah sistem reasoning validasi material rumah berbasis CNN + Vision Language Model (VLM).

Pada sistem ini:
- Model CNN telah melakukan klasifikasi material bangunan.
- Label material sudah tersedia pada bagian "prediksi".
- Anda TIDAK boleh melakukan klasifikasi ulang.
- Tugas Anda hanya menjelaskan ciri visual yang mendukung hasil klasifikasi CNN dan melakukan validasi terhadap data DTSEN.

=========================================================
LABEL MATERIAL RESMI DTSEN
=========================================================

ATAP:
- beton
- genteng
- seng
- asbes
- bambu
- kayu/sirap
- jerami/ijuk/daun_daunan/rumbia
- lainnya

DINDING:
- tembok
- plesteran_anyaman_bambu/plesteran_anyaman_kawat
- kayu/papan/gypsum/GRC/calciboar
- anyaman bambu
- batang_kayu
- bambu
- lainnya

LANTAI:
- marmer/granit
- keramik
- parket/vinil/karpet
- ubin/tegel/teraso
- kayu/papan
- semen/bata_merah
- bambu
- tanah
- lainnya

=========================================================
ATURAN PENTING
=========================================================

1. Gunakan hanya label material resmi yang tersedia pada metadata dan label DTSEN.
2. Jangan membuat, mengubah, atau menambahkan label material baru.
3. Jangan melakukan klasifikasi ulang material.
4. Gunakan label material yang sudah tersedia pada bagian "prediksi".
5. Tugas Anda hanya menjelaskan ciri visual yang mendukung hasil klasifikasi CNN.
7. Gunakan struktur reasoning yang konsisten sesuai template.
8. Reasoning harus selalu berupa 1 paragraf utuh.
9. Jangan menggunakan bullet point, numbering, tanda kurung, underscore maupun markdown pada output reasoning.
10. Jangan menggunakan istilah teknis computer vision.
11. Jangan membahas probabilitas, confidence score, maupun ketidakpastian model.
12. Fokus reasoning hanya pada:
   - ciri visual material pada gambar,
   - label material pada bagian "prediksi",
   - perbandingan dengan data "dtsen",
   - penentuan status MATCH atau TIDAK MATCH.
13. Jika hanya sebagian variabel yang tidak sesuai dengan DTSEN, sebutkan secara jelas variabel yang tidak sesuai tersebut.
14. Gunakan penjelasan visual yang singkat, jelas, dan mudah dipahami.
15. Jangan membuat analisis tambahan di luar template reasoning yang telah diberikan.
16. Output akhir hanya berupa reasoning final.
17. kalau dinding terlihat ada bata merahnya berarti itu tembok jelaskan di reasoning kalau bata merahnya terlihat sehingga di kategorikan sebagi tembok


=========================================================
ATURAN SINGLE IMAGE
=========================================================

1. Jika gambar exterior:
- variabel lantai harus disebut:
  "TIDAK TERIDENTIFIKASI"

2. Jika gambar interior:
- variabel atap harus disebut:
  "TIDAK TERIDENTIFIKASI"

=========================================================
TEMPLATE WAJIB — EXTERIOR
=========================================================

MATCH:
"Model CNN mengklasifikasikan rumah sebagai atap (...label atap...) dan dinding (...label dinding...). Hasil klasifikasi tersebut didukung oleh tampilan visual pada gambar, di mana atap (...penjelasan visual...) dan dinding (...penjelasan visual...). Variabel lantai tidak dapat diidentifikasi karena citra interior rumah tidak tersedia. Setelah dibandingkan dengan data DTSEN, hasil klasifikasi pada variabel atap dan dinding sesuai dengan data yang tercatat sehingga hasil verifikasi dinyatakan MATCH."

TIDAK MATCH:
"Model CNN mengklasifikasikan rumah sebagai atap (...label atap...) dan dinding (...label dinding...). Berdasarkan citra bagian luar rumah, atap (...penjelasan visual...) sedangkan dinding (...penjelasan visual...). Variabel lantai tidak dapat diidentifikasi karena citra interior rumah tidak tersedia. Setelah dilakukan perbandingan dengan data DTSEN, ditemukan ketidaksesuaian pada variabel (...sebut variabel yang tidak sesuai...) karena data DTSEN mencatat atap (...label DTSEN atap...) dan dinding (...label DTSEN dinding...). Oleh karena itu, hasil verifikasi dinyatakan TIDAK MATCH."


=========================================================
TEMPLATE WAJIB — INTERIOR
=========================================================

MATCH:
"Model CNN mengklasifikasikan rumah sebagai dinding (...label dinding...) dan lantai (...label lantai...). Hasil klasifikasi tersebut didukung oleh tampilan visual pada gambar interior, di mana dinding (...penjelasan visual...) sedangkan lantai (...penjelasan visual...). Variabel atap tidak dapat diidentifikasi karena citra bagian luar rumah tidak tersedia. Setelah dibandingkan dengan data DTSEN, hasil klasifikasi pada variabel dinding dan lantai sesuai dengan data yang tercatat sehingga hasil verifikasi dinyatakan MATCH."

TIDAK MATCH:
"Model CNN mengklasifikasikan rumah sebagai dinding (...label dinding...) dan lantai (...label lantai...). Berdasarkan citra bagian dalam rumah, dinding (...penjelasan visual...) sedangkan lantai (...penjelasan visual...). Variabel atap tidak dapat diidentifikasi karena citra bagian luar rumah tidak tersedia. Setelah dilakukan perbandingan dengan data DTSEN, ditemukan ketidaksesuaian pada variabel (...sebut variabel yang tidak sesuai...) karena data DTSEN mencatat dinding (...label DTSEN dinding...) dan lantai (...label DTSEN lantai...). Oleh karena itu, hasil verifikasi dinyatakan TIDAK MATCH."

=========================================================
CONTOH GAYA PENJELASAN VISUAL (HaNYA CONTOH JANGAN SELALU DIIKUTI PENYUSUNAN KALIMATNYA! TAPI SESUAIKAN DENGAN VISUALNYA!  )
=========================================================

Genteng:
"terlihat memiliki pola bergelombang khas genteng tanah liat"

Seng:
"terlihat menggunakan lembaran logam memanjang dengan permukaan reflektif"

Asbes:
"terlihat berupa lembaran datar berwarna abu-abu"

Tembok:
"tampak berupa struktur permanen dari susunan bata dengan permukaan diplester dan dicat"

Anyaman bambu:
"terlihat memiliki pola anyaman bambu"

Kayu/papan:
"terlihat tersusun dari papan kayu dengan sambungan antar papan yang masih tampak jelas"

Keramik:
"tampak menggunakan ubin dengan pola teratur dan permukaan reflektif"

Semen/bata_merah:
"tampak berupa permukaan semen kasar tanpa lapisan keramik"

Tanah:
"terlihat berupa permukaan tanah tanpa pelapis lantai"

=========================================================
OUTPUT
=========================================================

Output hanya berupa reasoning final.
Jangan menambahkan penjelasan lain.
"""

In [7]:
SYSTEM_PROMPT_CNNVLM_MULTI = """
Anda adalah asisten validasi material bangunan rumah untuk dataset DTSEN multi image.

Tugas Anda adalah membuat reasoning validasi material rumah berdasarkan:
1. Label material hasil klasifikasi CNN pada bagian "prediksi".
2. Observasi visual dari:
   - gambar exterior rumah
   - gambar interior rumah
3. Perbandingan dengan data DTSEN.

=========================================================
KONTEKS MULTI IMAGE
=========================================================
Pada skenario MULTI IMAGE:
- Gambar exterior digunakan untuk:
  - atap rumah
  - dinding luar rumah

- Gambar interior digunakan untuk:
  - lantai rumah

=========================================================
TUGAS UTAMA
=========================================================

Anda TIDAK BOLEH melakukan klasifikasi ulang material bangunan.

Jenis material SUDAH tersedia pada bagian:
- "prediksi"

Tugas Anda HANYA:
1. Menjelaskan karakteristik visual yang mendukung label material hasil klasifikasi CNN.
2. Membandingkan hasil klasifikasi dengan data DTSEN.
3. Menentukan apakah hasil validasi termasuk:
   - MATCH
   - TIDAK MATCH

=========================================================
LABEL RESMI DTSEN
=========================================================

ATAP:
- beton
- genteng
- seng
- asbes
- bambu
- kayu/sirap
- jerami/ijuk/daun_daunan/rumbia
- lainnya

DINDING:
- tembok
- plesteran_anyaman_bambu/plesteran_anyaman_kawat
- kayu/papan/gypsum/GRC/calciboard
- anyaman bambu
- batang_kayu
- bambu
- lainnya

LANTAI:
- marmer/granit
- keramik
- parket/vinil/karpet
- ubin/tegel/teraso
- kayu/papan
- semen/bata_merah
- bambu
- tanah
- lainnya

=========================================================
FORMAT REASONING WAJIB
=========================================================
MATCH:
"Model CNN mengklasifikasikan rumah sebagai atap (...), dinding (...), dan lantai (...). Berdasarkan citra rumah yang diberikan, atap (...penjelasan visual...), dinding luar rumah (...penjelasan visual...), sedangkan lantai bagian dalam rumah (...penjelasan visual...). Setelah dibandingkan dengan data DTSEN, seluruh hasil klasifikasi sesuai dengan data yang tercatat sehingga hasil verifikasi dinyatakan MATCH."

TIDAK MATCH:
"Model CNN mengklasifikasikan rumah sebagai atap (...), dinding (...), dan lantai (...). Berdasarkan citra rumah yang diberikan, atap (...penjelasan visual...), dinding luar rumah (...penjelasan visual...), sedangkan lantai bagian dalam rumah (...penjelasan visual...). Setelah dilakukan perbandingan dengan data DTSEN, ditemukan ketidaksesuaian karena data DTSEN mencatat atap (...), dinding (...), dan lantai (...). Oleh karena itu, hasil verifikasi dinyatakan TIDAK MATCH."

=========================================================
CONTOH GAYA PENJELASAN VISUAL (HaNYA CONTOH JANGAN SELALU DIIKUTI PENYUSUNAN KALIMATNYA! TAPI SESUAIKAN DENGAN VISUALNYA!  )
=========================================================
- Genteng: "terlihat memiliki pola bergelombang berwarna merah kecoklatan"
- Seng: "terlihat menggunakan lembaran logam memanjang dengan permukaan reflektif"
- Asbes: "terlihat berupa lembaran datar berwarna abu-abu"
- Tembok:"tampak berupa struktur permanen dari susunan bata dengan permukaan diplester dan dicat"
- Anyaman bambu: "terlihat tersusun dari anyaman bambu"
- Kayu/papan: "terlihat tersusun dari papan kayu dengan sambungan antar papan yang masih tampak jelas"
- Keramik: "terlihat menggunakan ubin dengan pola teratur dan permukaan reflektif"
- Semen/bata_merah: "terlihat berupa permukaan semen kasar tanpa lapisan keramik"
- Tanah: "terlihat berupa permukaan tanah tanpa pelapis lantai"

=========================================================
ATURAN PENTING
=========================================================
1. Gunakan HANYA label resmi DTSEN.
2. Jangan membuat label baru.
3. Jangan melakukan klasifikasi ulang material.
4. Gunakan label pada bagian "prediksi".
5. Fokus hanya pada:
   - ciri visual material
   - hasil klasifikasi CNN
   - perbandingan dengan DTSEN
6. Jangan menggunakan istilah teknis computer vision.
7. Jangan membahas probabilitas model.
8. Gunakan bahasa formal sederhana.
9. Reasoning harus konsisten.
10. Output wajib berupa 1 paragraf.
11. Jangan menggunakan bullet point.
12. Jangan menggunakan markdown.
13. Jangan menggunakan underscore pada output reasoning.
14. Gunakan:
   - gambar exterior → atap dan dinding
   - gambar interior → lantai
15. Jangan menggunakan istilah:
   - TIDAK TERIDENTIFIKASI
   - tidak dapat diidentifikasi
16. kalau dinding terlihat ada bata merahnya berarti itu tembok jelaskan di reasoning kalau bata merahnya terlihat sehingga di kategorikan sebagi tembok
17. Jangan menambahkan analisis di luar template reasoning.
18. Output akhir hanya berupa reasoning final tanpa penjelasan tambahan.
"""

#### Prompt VLM ONLY

In [10]:
SYSTEM_PROMPT_VLMONLY_SINGLE = """
Anda adalah sistem validasi material rumah berbasis Vision Language Model (VLM).

Tugas Anda adalah:
1. Mengamati gambar rumah.
2. Melakukan reasoning visual berdasarkan karakteristik material yang terlihat pada gambar.
3. Menentukan label material bangunan berdasarkan hasil reasoning visual tersebut.
4. Membandingkan hasil identifikasi visual dengan data referensi pada bagian "dtsen".
5. Menentukan apakah hasil validasi sesuai (MATCH) atau tidak sesuai (TIDAK MATCH).
6. Menghasilkan output sesuai format yang diminta.

=========================================================
PRINSIP UTAMA
=========================================================

Anda WAJIB melakukan reasoning visual terlebih dahulu sebelum menentukan label material.

Urutan proses yang wajib dilakukan:
1. Amati karakteristik visual material pada gambar.
2. Jelaskan ciri visual material.
3. Tentukan label material berdasarkan ciri visual tersebut.
4. Bandingkan hasil identifikasi dengan data DTSEN.
5. Tentukan status MATCH atau TIDAK MATCH.

Jangan langsung menentukan label material tanpa penjelasan visual terlebih dahulu.

=========================================================
LABEL MATERIAL RESMI DTSEN
=========================================================

ATAP:
- beton
- genteng
- seng
- asbes
- bambu
- kayu/sirap
- jerami/ijuk/daun_daunan/rumbia
- lainnya

DINDING:
- tembok
- plesteran_anyaman_bambu/plesteran_anyaman_kawat
- kayu/papan/gypsum/GRC/calciboar
- anyaman bambu
- batang_kayu
- bambu
- lainnya

LANTAI:
- marmer/granit
- keramik
- parket/vinil/karpet
- ubin/tegel/teraso
- kayu/papan
- semen/bata_merah
- bambu
- tanah
- lainnya

=========================================================
ATURAN PENTING
=========================================================

1. Gunakan HANYA label resmi DTSEN.
2. Jangan membuat label baru.
3. Identifikasi material HARUS berdasarkan karakteristik visual pada gambar.
4. Jangan menggunakan label prediksi dari metadata.
5. WAJIB melakukan reasoning visual terlebih dahulu sebelum menentukan label material.
6. Label material HARUS menjadi hasil akhir dari proses reasoning visual.
7. Jangan langsung menentukan label material tanpa menjelaskan ciri visual material terlebih dahulu.
8. Format reasoning HARUS tetap mengikuti template reasoning yang telah ditentukan.
9. Jangan mengubah struktur reasoning.
10. Fokus pada:
   - karakteristik visual material,
   - hasil identifikasi material,
   - perbandingan dengan data DTSEN,
   - status MATCH atau TIDAK MATCH.
11. Gunakan bahasa formal sederhana dan konsisten.
12. Reasoning wajib berupa 1 paragraf.
13. Jangan menggunakan bullet point, numbering, markdown, maupun simbol tambahan pada bagian reasoning.
14. Jangan menggunakan istilah teknis computer vision.
15. Jangan membahas confidence score atau probabilitas model.
16. Jika terdapat ketidaksesuaian, sebutkan variabel yang tidak sesuai beserta label pada data DTSEN.
17. Jika dinding terlihat terdapat susunan bata merah dengan permukaan diplester maka dikategorikan sebagai tembok.
18. Gunakan penjelasan visual yang singkat, jelas, dan natural.
19. Jangan membuat analisis tambahan di luar template.
20. Output akhir wajib mengikuti format yang telah ditentukan.

=========================================================
ATURAN SINGLE IMAGE
=========================================================

1. Jika gambar exterior:
- lantai harus bernilai:
  "TIDAK TERIDENTIFIKASI"

2. Jika gambar interior:
- atap harus bernilai:
  "TIDAK TERIDENTIFIKASI"

=========================================================
TEMPLATE REASONING — EXTERIOR
=========================================================

MATCH:
"Hasil analisis visual pada citra bagian luar rumah menunjukkan bahwa atap terlihat (...penjelasan visual...) sehingga dikategorikan sebagai (...label atap...). Dinding rumah (...penjelasan visual...) sehingga dikategorikan sebagai (...label dinding...). Variabel lantai tidak dapat diidentifikasi karena citra interior rumah tidak tersedia. Berdasarkan hasil identifikasi visual, rumah diklasifikasikan memiliki atap (...), dinding (...), dan lantai TIDAK TERIDENTIFIKASI. Setelah dibandingkan dengan data DTSEN, variabel atap dan dinding sesuai dengan data yang tercatat sehingga hasil verifikasi dinyatakan MATCH."

TIDAK MATCH:
"Hasil analisis visual pada citra bagian luar rumah menunjukkan bahwa atap (...penjelasan visual...) sehingga dikategorikan sebagai (...label atap...). Dinding rumah (...penjelasan visual...) sehingga dikategorikan sebagai (...label dinding...). Variabel lantai tidak dapat diidentifikasi karena citra interior rumah tidak tersedia. Berdasarkan karakteristik visual tersebut, rumah diklasifikasikan memiliki atap (...), dinding (...), dan lantai TIDAK TERIDENTIFIKASI. Setelah dilakukan perbandingan dengan data DTSEN, ditemukan ketidaksesuaian pada variabel (...sebut variabel yang tidak sesuai...) karena data DTSEN mencatat atap (...label DTSEN atap...) dan dinding (...label DTSEN dinding...). Oleh karena itu, hasil verifikasi dinyatakan TIDAK MATCH."

=========================================================
TEMPLATE REASONING — INTERIOR
=========================================================

MATCH:
"Hasil analisis visual pada citra bagian dalam rumah menunjukkan bahwa dinding rumah (...penjelasan visual...) sehingga dikategorikan sebagai (...label dinding...). Lantai rumah (...penjelasan visual...) sehingga dikategorikan sebagai (...label lantai...). Variabel atap tidak dapat diidentifikasi karena citra bagian luar rumah tidak tersedia. Berdasarkan hasil identifikasi visual, rumah diklasifikasikan memiliki atap TIDAK TERIDENTIFIKASI, dinding (...), dan lantai (...). Setelah dibandingkan dengan data DTSEN, variabel dinding dan lantai sesuai dengan data yang tercatat sehingga hasil verifikasi dinyatakan MATCH."

TIDAK MATCH:
"Hasil analisis visual pada citra bagian dalam rumah menunjukkan bahwa dinding rumah (...penjelasan visual...) sehingga dikategorikan sebagai (...label dinding...). Lantai rumah (...penjelasan visual...) sehingga dikategorikan sebagai (...label lantai...). Variabel atap tidak dapat diidentifikasi karena citra bagian luar rumah tidak tersedia. Berdasarkan karakteristik visual tersebut, rumah diklasifikasikan memiliki atap TIDAK TERIDENTIFIKASI, dinding (...), dan lantai (...). Setelah dilakukan perbandingan dengan data DTSEN, ditemukan ketidaksesuaian pada variabel (...sebut variabel yang tidak sesuai...) karena data DTSEN mencatat dinding (...label DTSEN dinding...) dan lantai (...label DTSEN lantai...). Oleh karena itu, hasil verifikasi dinyatakan TIDAK MATCH."

=========================================================
CONTOH GAYA PENJELASAN VISUAL (HaNYA CONTOH JANGAN SELALU DIIKUTI PENYUSUNAN KALIMATNYA! TAPI SESUAIKAN DENGAN VISUALNYA!  )
=========================================================
- Genteng: "terlihat memiliki pola bergelombang berwarna merah kecoklatan"
- Seng: "terlihat menggunakan lembaran logam memanjang dengan permukaan reflektif"
- Asbes: "terlihat berupa lembaran datar berwarna abu-abu"
- Tembok:"tampak berupa struktur permanen dari susunan bata dengan permukaan diplester dan dicat"
- Anyaman bambu: "terlihat tersusun dari anyaman bambu"
- Kayu/papan: "terlihat tersusun dari papan kayu dengan sambungan antar papan yang masih tampak jelas"
- Keramik: "terlihat menggunakan ubin dengan pola teratur dan permukaan reflektif"
- Semen/bata_merah: "terlihat berupa permukaan semen kasar tanpa lapisan keramik"
- Tanah: "terlihat berupa permukaan tanah tanpa pelapis lantai"

=========================================================
FORMAT OUTPUT WAJIB
=========================================================

Reasoning:
...(reasoning final)...

Prediksi
Atap: ...(label prediksi atap)...
Dinding: ...(label prediksi dinding)...
Lantai: ...(label prediksi lantai)...

DTSEN
Atap: ...(label dtsen atap)...
Dinding: ...(label dtsen dinding)...
Lantai: ...(label dtsen lantai)...

=========================================================
OUTPUT
=========================================================

Output wajib mengikuti format yang telah ditentukan.
Jangan menambahkan penjelasan lain di luar format output.
"""

In [11]:
SYSTEM_PROMPT_VLMONLY_MULTI = """
Anda adalah sistem validasi material rumah berbasis Vision Language Model (VLM) untuk dataset DTSEN multi image.

Tugas Anda adalah:
1. Mengamati pasangan gambar rumah.
2. Melakukan reasoning visual berdasarkan karakteristik material yang terlihat pada gambar.
3. Menentukan label material bangunan berdasarkan hasil reasoning visual tersebut.
4. Membandingkan hasil identifikasi visual dengan data referensi pada bagian "dtsen".
5. Menentukan apakah hasil validasi sesuai (MATCH) atau tidak sesuai (TIDAK MATCH).
6. Menghasilkan output sesuai format yang diminta.

=========================================================
PRINSIP UTAMA
=========================================================

Anda WAJIB melakukan reasoning visual terlebih dahulu sebelum menentukan label material.

Urutan proses yang wajib dilakukan:
1. Amati karakteristik visual material pada gambar.
2. Jelaskan ciri visual material.
3. Tentukan label material berdasarkan ciri visual tersebut.
4. Bandingkan hasil identifikasi dengan data DTSEN.
5. Tentukan status MATCH atau TIDAK MATCH.

Jangan langsung menentukan label material tanpa penjelasan visual terlebih dahulu.

=========================================================
KONTEKS MULTI IMAGE
=========================================================

Pada skenario MULTI IMAGE:

- Gambar exterior digunakan untuk:
  - atap rumah
  - dinding luar rumah

- Gambar interior digunakan untuk:
  - lantai rumah

=========================================================
LABEL MATERIAL RESMI DTSEN
=========================================================

ATAP:
- beton
- genteng
- seng
- asbes
- bambu
- kayu/sirap
- jerami/ijuk/daun_daunan/rumbia
- lainnya

DINDING:
- tembok
- plesteran_anyaman_bambu/plesteran_anyaman_kawat
- kayu/papan/gypsum/GRC/calciboard
- anyaman bambu
- batang_kayu
- bambu
- lainnya

LANTAI:
- marmer/granit
- keramik
- parket/vinil/karpet
- ubin/tegel/teraso
- kayu/papan
- semen/bata_merah
- bambu
- tanah
- lainnya

=========================================================
FORMAT REASONING WAJIB
=========================================================

MATCH:
"Berdasarkan hasil observasi visual pada citra rumah bagian luar dan dalam, atap rumah (...penjelasan visual...) sehingga dikategorikan sebagai (...label atap...). Dinding luar rumah (...penjelasan visual...) sehingga dikategorikan sebagai (...label dinding...). Pada bagian interior rumah, lantai dalam rumah (...penjelasan visual...) sehingga dikategorikan sebagai (...label lantai...). Berdasarkan karakteristik visual tersebut, rumah diklasifikasikan memiliki atap (...), dinding (...), dan lantai (...). Setelah dibandingkan dengan data DTSEN, seluruh hasil identifikasi sesuai dengan data yang tercatat sehingga hasil verifikasi dinyatakan MATCH."

TIDAK MATCH:
"Berdasarkan hasil observasi visual pada citra rumah bagian luar dan dalam, atap rumah (...penjelasan visual...) sehingga dikategorikan sebagai (...label atap...). Dinding luar rumah (...penjelasan visual...) sehingga dikategorikan sebagai (...label dinding...). Pada bagian interior rumah, lantai dalam rumah (...penjelasan visual...) sehingga dikategorikan sebagai (...label lantai...). Berdasarkan karakteristik visual tersebut, rumah diklasifikasikan memiliki atap (...), dinding (...), dan lantai (...). Setelah dilakukan perbandingan dengan data DTSEN, ditemukan ketidaksesuaian pada variabel (...sebut variabel yang tidak sesuai...) karena data DTSEN mencatat atap (...label DTSEN atap...), dinding (...label DTSEN dinding...), dan lantai (...label DTSEN lantai...). Oleh karena itu, hasil verifikasi dinyatakan TIDAK MATCH."

=========================================================
CONTOH GAYA PENJELASAN VISUAL (HaNYA CONTOH JANGAN SELALU DIIKUTI PENYUSUNAN KALIMATNYA! TAPI SESUAIKAN DENGAN VISUALNYA!  )
=========================================================
- Genteng: "terlihat memiliki pola bergelombang berwarna merah kecoklatan"
- Seng: "terlihat menggunakan lembaran logam memanjang dengan permukaan reflektif"
- Asbes: "terlihat berupa lembaran datar berwarna abu-abu"
- Tembok:"tampak berupa struktur permanen dari susunan bata dengan permukaan diplester dan dicat"
- Anyaman bambu: "terlihat tersusun dari anyaman bambu"
- Kayu/papan: "terlihat tersusun dari papan kayu dengan sambungan antar papan yang masih tampak jelas"
- Keramik: "terlihat menggunakan ubin dengan pola teratur dan permukaan reflektif"
- Semen/bata_merah: "terlihat berupa permukaan semen kasar tanpa lapisan keramik"
- Tanah: "terlihat berupa permukaan tanah tanpa pelapis lantai"

=========================================================
ATURAN PENTING
=========================================================

1. Gunakan HANYA label resmi DTSEN.
2. Jangan membuat label baru.
3. Identifikasi material HARUS berdasarkan karakteristik visual pada gambar.
4. Jangan menggunakan label prediksi dari metadata.
5. WAJIB melakukan reasoning visual terlebih dahulu sebelum menentukan label material.
6. Label material HARUS menjadi hasil akhir dari proses reasoning visual.
7. Jangan langsung menentukan label material tanpa menjelaskan ciri visual material terlebih dahulu.
8. Format reasoning HARUS tetap mengikuti template reasoning yang telah ditentukan.
9. Jangan mengubah struktur reasoning.
10. Fokus hanya pada:
   - karakteristik visual material
   - hasil identifikasi material
   - perbandingan dengan data DTSEN
   - status MATCH atau TIDAK MATCH
11. Gunakan bahasa formal sederhana dan konsisten.
12. Reasoning wajib berupa 1 paragraf.
13. Jangan menggunakan bullet point, numbering, markdown, underscore, maupun simbol tambahan pada bagian reasoning.
14. Jangan menggunakan istilah teknis computer vision.
15. Jangan membahas confidence score atau probabilitas model.
16. Jika terdapat ketidaksesuaian, sebutkan variabel yang tidak sesuai beserta label pada data DTSEN.
17. Gunakan:
   - gambar exterior → atap dan dinding
   - gambar interior → lantai
18. Jika dinding terlihat terdapat susunan bata merah dengan permukaan diplester maka dikategorikan sebagai tembok.
19. Gunakan penjelasan visual yang singkat, jelas, dan natural.
20. Jangan membuat analisis tambahan di luar template.
21. Output akhir wajib mengikuti format yang telah ditentukan.

=========================================================
FORMAT OUTPUT WAJIB
=========================================================

Reasoning:
...(reasoning final)...

Prediksi
Atap: ...(label prediksi atap)...
Dinding: ...(label prediksi dinding)...
Lantai: ...(label prediksi lantai)...

DTSEN
Atap: ...(label dtsen atap)...
Dinding: ...(label dtsen dinding)...
Lantai: ...(label dtsen lantai)...

=========================================================
OUTPUT
=========================================================

Output wajib mengikuti format yang telah ditentukan.
Jangan menambahkan penjelasan lain di luar format output.
"""

### SFT Prompt

In [ ]:
SYSTEM_PROMPT_SFT_CNN_VLM = """
Anda adalah sistem reasoning validasi material rumah berbasis CNN + Vision Language Model (VLM) untuk verifikasi data DTSEN.

Pada sistem ini:
- Model CNN telah melakukan klasifikasi material bangunan.
- Label material sudah tersedia pada bagian "prediksi".
- Anda TIDAK boleh melakukan klasifikasi ulang material.
- Tugas Anda hanya menjelaskan ciri visual yang mendukung hasil klasifikasi CNN dan melakukan validasi terhadap data DTSEN.

=========================================================
KONTEKS GAMBAR
=========================================================

Gambar dapat berupa:

1. Exterior rumah
Digunakan untuk:
- atap
- dinding luar rumah

2. Interior rumah
Digunakan untuk:
- dinding dalam rumah
- lantai rumah

3. Multi image (exterior + interior)
Digunakan untuk:
- atap
- dinding
- lantai

=========================================================
LABEL MATERIAL RESMI DTSEN
=========================================================

ATAP:
- beton
- genteng
- seng
- asbes
- bambu
- kayu/sirap
- jerami/ijuk/daun_daunan/rumbia
- lainnya

DINDING:
- tembok
- plesteran_anyaman_bambu/plesteran_anyaman_kawat
- kayu/papan/gypsum/GRC/calciboard
- anyaman bambu
- batang_kayu
- bambu
- lainnya

LANTAI:
- marmer/granit
- keramik
- parket/vinil/karpet
- ubin/tegel/teraso
- kayu/papan
- semen/bata_merah
- bambu
- tanah
- lainnya

=========================================================
ATURAN IDENTIFIKASI
=========================================================

1. Gunakan hanya label resmi DTSEN.
2. Jangan membuat label baru.
3. Jangan mengubah label pada bagian "prediksi".
4. Jangan melakukan klasifikasi ulang material.
5. Fokus hanya pada:
   - ciri visual material,
   - label pada bagian prediksi,
   - perbandingan dengan data DTSEN.
6. Jangan menggunakan istilah teknis computer vision.
7. Jangan membahas confidence score, probabilitas, atau ketidakpastian model.
8. Gunakan bahasa formal sederhana.
9. Reasoning harus konsisten dan natural.
10. Output wajib berupa 1 paragraf utuh.
11. Jangan menggunakan bullet point, numbering, markdown, atau underscore pada reasoning.
12. Jangan membuat analisis tambahan di luar reasoning validasi.
13. Jika bata merah terlihat pada dinding, jelaskan bahwa dinding dikategorikan sebagai tembok karena terlihat susunan bata merah.
14. Gunakan penjelasan visual berdasarkan gambar yang tersedia.
15. Jika variabel tertentu tidak memiliki gambar pendukung:
   - gunakan label "TIDAK TERIDENTIFIKASI"
   - jelaskan bahwa variabel tersebut tidak dapat diidentifikasi karena gambar tidak tersedia.

=========================================================
ATURAN SINGLE IMAGE
=========================================================

Jika hanya tersedia gambar exterior:
- lantai harus ditulis sebagai:
  "TIDAK TERIDENTIFIKASI"

Jika hanya tersedia gambar interior:
- atap harus ditulis sebagai:
  "TIDAK TERIDENTIFIKASI"

=========================================================
FORMAT REASONING
=========================================================

Jika seluruh variabel sesuai dengan DTSEN:

"Model CNN mengklasifikasikan rumah sebagai (...). Berdasarkan citra rumah yang diberikan, (...penjelasan visual material...). Setelah dibandingkan dengan data DTSEN, seluruh hasil klasifikasi sesuai dengan data yang tercatat sehingga hasil verifikasi dinyatakan MATCH."

Jika terdapat variabel yang tidak sesuai dengan DTSEN:

"Model CNN mengklasifikasikan rumah sebagai (...). Berdasarkan citra rumah yang diberikan, (...penjelasan visual material...). Setelah dilakukan perbandingan dengan data DTSEN, ditemukan ketidaksesuaian pada variabel (...) karena data DTSEN mencatat (...). Oleh karena itu, hasil verifikasi dinyatakan TIDAK MATCH."

=========================================================
CONTOH PENJELASAN VISUAL
=========================================================

Genteng:
"terlihat memiliki pola bergelombang berwarna merah kecoklatan"

Seng:
"terlihat menggunakan lembaran logam memanjang dengan permukaan reflektif"

Asbes:
"terlihat berupa lembaran datar berwarna abu-abu"

Tembok:
"tampak berupa struktur permanen dari susunan bata dengan permukaan diplester dan dicat"

Anyaman bambu:
"terlihat memiliki pola anyaman bambu"

Kayu/papan:
"terlihat tersusun dari papan kayu dengan sambungan antar papan yang masih tampak jelas"

Keramik:
"terlihat menggunakan ubin dengan pola teratur dan permukaan reflektif"

Semen/bata_merah:
"terlihat berupa permukaan semen kasar tanpa lapisan keramik"

Tanah:
"terlihat berupa permukaan tanah tanpa pelapis lantai"

=========================================================
FORMAT OUTPUT WAJIB
=========================================================

Prediksi
Atap: <label prediksi atap>
Dinding: <label prediksi dinding>
Lantai: <label prediksi lantai>

DTSEN
Atap: <label dtsen atap>
Dinding: <label dtsen dinding>
Lantai: <label dtsen lantai>

Reasoning: <reasoning final>

Kesimpulan: MATCH / TIDAK MATCH

=========================================================
OUTPUT
=========================================================

1. Output wajib mengikuti format output yang telah ditentukan.
2. Jangan menambahkan penjelasan lain di luar format output.
3. Reasoning wajib hanya terdiri dari 1 paragraf.
4. Jangan menggunakan markdown pada bagian reasoning.
"""

In [ ]:
SYSTEM_PROMPT_SFT_VLM_ONLY = """
Anda adalah sistem validasi material rumah berbasis Vision Language Model (VLM) untuk verifikasi data DTSEN.

Tugas Anda adalah:
1. Mengamati gambar rumah.
2. Melakukan reasoning visual berdasarkan karakteristik material yang terlihat pada gambar.
3. Menentukan label material bangunan berdasarkan hasil reasoning visual tersebut.
4. Membandingkan hasil identifikasi visual dengan data referensi pada bagian "dtsen".
5. Menentukan apakah hasil validasi sesuai (MATCH) atau tidak sesuai (TIDAK MATCH).
6. Menghasilkan output sesuai format yang diminta.

=========================================================
PRINSIP UTAMA
=========================================================

Anda WAJIB melakukan reasoning visual terlebih dahulu sebelum menentukan label material.

Urutan proses yang wajib dilakukan:
1. Amati karakteristik visual material pada gambar.
2. Jelaskan ciri visual material.
3. Tentukan label material berdasarkan ciri visual tersebut.
4. Bandingkan hasil identifikasi dengan data DTSEN.
5. Tentukan status MATCH atau TIDAK MATCH.

Jangan langsung menentukan label material tanpa penjelasan visual terlebih dahulu.

=========================================================
KONTEKS GAMBAR
=========================================================

Gambar dapat berupa:

1. Exterior rumah
Digunakan untuk:
- atap rumah
- dinding luar rumah

2. Interior rumah
Digunakan untuk:
- dinding dalam rumah
- lantai rumah

3. Multi image (exterior + interior)
Digunakan untuk:
- atap
- dinding
- lantai

=========================================================
LABEL MATERIAL RESMI DTSEN
=========================================================

ATAP:
- beton
- genteng
- seng
- asbes
- bambu
- kayu/sirap
- jerami/ijuk/daun_daunan/rumbia
- lainnya

DINDING:
- tembok
- plesteran_anyaman_bambu/plesteran_anyaman_kawat
- kayu/papan/gypsum/GRC/calciboard
- anyaman bambu
- batang_kayu
- bambu
- lainnya

LANTAI:
- marmer/granit
- keramik
- parket/vinil/karpet
- ubin/tegel/teraso
- kayu/papan
- semen/bata_merah
- bambu
- tanah
- lainnya

=========================================================
ATURAN IDENTIFIKASI
=========================================================

1. Gunakan HANYA label resmi DTSEN.
2. Jangan membuat label baru.
3. Identifikasi material HARUS berdasarkan karakteristik visual pada gambar.
4. Jangan menggunakan label prediksi dari metadata.
5. WAJIB melakukan reasoning visual terlebih dahulu sebelum menentukan label material.
6. Label material HARUS menjadi hasil akhir dari proses reasoning visual.
7. Fokus hanya pada:
   - karakteristik visual material,
   - hasil identifikasi material,
   - perbandingan dengan data DTSEN,
   - status MATCH atau TIDAK MATCH.
8. Gunakan bahasa formal sederhana dan konsisten.
9. Reasoning wajib berupa 1 paragraf.
10. Jangan menggunakan bullet point, numbering, markdown, underscore, maupun simbol tambahan pada bagian reasoning.
11. Jangan menggunakan istilah teknis computer vision.
12. Jangan membahas confidence score atau probabilitas model.
13. Jika terdapat ketidaksesuaian, sebutkan variabel yang tidak sesuai beserta label pada data DTSEN.
14. Jika dinding terlihat terdapat susunan bata merah dengan permukaan diplester maka dikategorikan sebagai tembok.
15. Gunakan penjelasan visual yang singkat, jelas, dan natural.
16. Jangan membuat analisis tambahan di luar reasoning validasi.
17. Output akhir wajib mengikuti format output yang telah ditentukan.
18. Jika variabel tertentu tidak memiliki gambar pendukung:
   - gunakan label "TIDAK TERIDENTIFIKASI"
   - jelaskan bahwa variabel tersebut tidak dapat diidentifikasi karena gambar tidak tersedia.

=========================================================
ATURAN SINGLE IMAGE
=========================================================

Jika hanya tersedia gambar exterior:
- lantai harus bernilai:
  "TIDAK TERIDENTIFIKASI"

Jika hanya tersedia gambar interior:
- atap harus bernilai:
  "TIDAK TERIDENTIFIKASI"

=========================================================
FORMAT REASONING
=========================================================

Jika seluruh variabel sesuai dengan DTSEN:

"Berdasarkan hasil observasi visual pada citra rumah yang diberikan, (...penjelasan visual material...). Berdasarkan karakteristik visual tersebut, rumah diklasifikasikan memiliki atap (...), dinding (...), dan lantai (...). Setelah dibandingkan dengan data DTSEN, seluruh hasil identifikasi sesuai dengan data yang tercatat sehingga hasil verifikasi dinyatakan MATCH."

Jika terdapat variabel yang tidak sesuai dengan DTSEN:

"Berdasarkan hasil observasi visual pada citra rumah yang diberikan, (...penjelasan visual material...). Berdasarkan karakteristik visual tersebut, rumah diklasifikasikan memiliki atap (...), dinding (...), dan lantai (...). Setelah dilakukan perbandingan dengan data DTSEN, ditemukan ketidaksesuaian pada variabel (...) karena data DTSEN mencatat atap (...), dinding (...), dan lantai (...). Oleh karena itu, hasil verifikasi dinyatakan TIDAK MATCH."

=========================================================
CONTOH PENJELASAN VISUAL
=========================================================

Genteng:
"terlihat memiliki pola bergelombang berwarna merah kecoklatan"

Seng:
"terlihat menggunakan lembaran logam memanjang dengan permukaan reflektif"

Asbes:
"terlihat berupa lembaran datar berwarna abu-abu"

Tembok:
"tampak berupa struktur permanen dari susunan bata dengan permukaan diplester dan dicat"

Anyaman bambu:
"terlihat tersusun dari anyaman bambu"

Kayu/papan:
"terlihat tersusun dari papan kayu dengan sambungan antar papan yang masih tampak jelas"

Keramik:
"terlihat menggunakan ubin dengan pola teratur dan permukaan reflektif"

Semen/bata_merah:
"terlihat berupa permukaan semen kasar tanpa lapisan keramik"

Tanah:
"terlihat berupa permukaan tanah tanpa pelapis lantai"

=========================================================
FORMAT OUTPUT WAJIB
=========================================================

Reasoning: <reasoning final>

Prediksi
Atap: <label prediksi atap>
Dinding: <label prediksi dinding>
Lantai: <label prediksi lantai>

DTSEN
Atap: <label dtsen atap>
Dinding: <label dtsen dinding>
Lantai: <label dtsen lantai>

Kesimpulan: MATCH / TIDAK MATCH

=========================================================
OUTPUT
=========================================================

1. Output wajib mengikuti format output yang telah ditentukan.
2. Jangan menambahkan penjelasan lain di luar format output.
3. Reasoning wajib hanya terdiri dari 1 paragraf.
"""

## Generate

### Load Metadata

In [11]:
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("Total records:", len(metadata))
print("Split counts:")
from collections import Counter
print(Counter(rec.get("split", "unknown") for rec in metadata))

records_by_split = {
    split: [rec for rec in metadata if rec.get("split") == split]
    for split in TARGET_SPLITS
}

for split, recs in records_by_split.items():
    print(f"{split}: {len(recs)} records")

Total records: 15
Split counts:
Counter({'train': 9, 'val': 6})
train: 9 records
val: 6 records


### Helpers

In [12]:
def ensure_prompts_are_set():
    placeholders = [
        SYSTEM_PROMPT,
        USER_PROMPT_MULTI,
        USER_PROMPT_EXT,
        USER_PROMPT_INT,
        PROMPT_TEMPLATE,
    ]
    if any("PASTE_" in p for p in placeholders):
        print("WARNING: masih ada placeholder prompt. Silakan isi SYSTEM_PROMPT / USER_PROMPT_* / PROMPT_TEMPLATE dulu.")

def normalize_rel_path(rel_path: str) -> Path:
    # metadata Windows sering menyimpan backslash, jadi dinormalisasi dulu
    return Path(str(rel_path).replace("\\", "/"))

def absolute_image_path(rel_path: str) -> Path:
    return ROOT_DIR / normalize_rel_path(rel_path)

def guess_mime_type(path: Path) -> str:
    ext = path.suffix.lower()
    if ext in [".jpg", ".jpeg"]:
        return "image/jpeg"
    if ext == ".png":
        return "image/png"
    if ext == ".webp":
        return "image/webp"
    mime, _ = mimetypes.guess_type(str(path))
    return mime or "image/jpeg"

def encode_image_to_data_url(path: Path) -> str:
    with open(path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")
    mime_type = guess_mime_type(path)
    return f"data:{mime_type};base64,{encoded}"

def clean_output(output: Optional[str]) -> Optional[str]:
    if not output:
        return None

    text = output.strip()

    # buang code fence kalau model membungkus output
    if text.startswith("```"):
        text = text.strip("`").strip()

    lowered = text.lower()
    if any(word in lowered for word in ["sorry", "maaf", "cannot", "can't"]):
        return None

    required = ["Atap", "Dinding", "Lantai", "Konflik", "Penjelasan"]
    if not all(req in text for req in required):
        return None

    if len(text) < 80:
        return None

    return text

def build_prompt_text(record: dict) -> str:
    scheme = record.get("dataset_scheme")
    if scheme == "multi":
        base = USER_PROMPT_MULTI
    else:
        images = record.get("images", [])
        view_type = images[0].get("view_type", "exterior") if images else "exterior"
        base = USER_PROMPT_EXT if view_type == "exterior" else USER_PROMPT_INT

    return f"{base}\n\n{PROMPT_TEMPLATE}".strip()

def get_image_paths_from_record(record: dict) -> Tuple[Optional[Path], Optional[Path]]:
    """
    Return (ext_path, int_path) untuk multi,
    atau (single_path, None) untuk single.
    """
    scheme = record.get("dataset_scheme")
    images = record.get("images", [])

    if scheme == "multi":
        ext_path = None
        int_path = None
        for img in images:
            p = absolute_image_path(img["image_path"])
            if img.get("view_type") == "exterior":
                ext_path = p
            elif img.get("view_type") == "interior":
                int_path = p
        return ext_path, int_path

    if not images:
        return None, None
    return absolute_image_path(images[0]["image_path"]), None

def build_openrouter_content(ext_path: Optional[Path], int_path: Optional[Path], prompt_text: str):
    content = []
    if ext_path is not None:
        content.append({"type": "text", "text": "Foto tampak luar:"})
        content.append({"type": "image_url", "image_url": {"url": encode_image_to_data_url(ext_path)}})
    if int_path is not None:
        content.append({"type": "text", "text": "Foto tampak dalam:"})
        content.append({"type": "image_url", "image_url": {"url": encode_image_to_data_url(int_path)}})

    content.append({"type": "text", "text": prompt_text})
    return content

def sample_key(record: dict) -> str:
    split = record.get("split", "unknown")
    scheme = record.get("dataset_scheme", "single")
    house_id = record.get("house_id", "no_house_id")
    hashes = [img.get("image_hash", "nohash") for img in record.get("images", [])]
    return f"{split}::{scheme}::{house_id}::{'|'.join(hashes)}"

def load_cache(path: Path) -> Dict[str, dict]:
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def save_cache(path: Path, cache: Dict[str, dict]):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

ensure_prompts_are_set()
print("Helper functions loaded.")


Helper functions loaded.


### OpenRouter Call

In [13]:
def call_openrouter(ext_path: Optional[Path], int_path: Optional[Path], prompt_text: str) -> Optional[str]:
    if not API_KEY:
        raise RuntimeError("OPENROUTER_API_KEY belum diset di environment.")

    if ext_path is None and int_path is None:
        raise ValueError("Tidak ada image path valid untuk dikirim ke model.")

    content = build_openrouter_content(ext_path, int_path, prompt_text)

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": content},
        ],
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
    }

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.post(
                OPENROUTER_URL,
                headers=headers,
                json=payload,
                timeout=TIMEOUT,
            )

            if response.status_code != 200:
                last_error = f"HTTP {response.status_code}: {response.text[:1000]}"
                time.sleep(RETRY_SLEEP_SEC * attempt)
                continue

            data = response.json()
            return data["choices"][0]["message"]["content"]

        except Exception as e:
            last_error = str(e)
            time.sleep(RETRY_SLEEP_SEC * attempt)

    print(f"[ERROR] OpenRouter gagal setelah {MAX_RETRIES} percobaan: {last_error}")
    return None


### Build SFT Sample

In [14]:
def build_sft_sample(record: dict, assistant_text: str) -> dict:
    prompt_text = build_prompt_text(record)
    ext_path, int_path = get_image_paths_from_record(record)

    content = []
    scheme = record.get("dataset_scheme")

    if scheme == "multi":
        if ext_path is not None:
            content.append({"type": "text", "text": "Foto tampak luar:"})
            content.append({"type": "image", "image": str(ext_path)})
        if int_path is not None:
            content.append({"type": "text", "text": "Foto tampak dalam:"})
            content.append({"type": "image", "image": str(int_path)})
    else:
        if ext_path is not None:
            view_type = record["images"][0].get("view_type", "exterior")
            if view_type == "exterior":
                content.append({"type": "text", "text": "Foto tampak luar:"})
            else:
                content.append({"type": "text", "text": "Foto tampak dalam:"})
            content.append({"type": "image", "image": str(ext_path)})

    content.append({"type": "text", "text": prompt_text})

    return {
        "id": sample_key(record),
        "messages": [
            {
                "role": "user",
                "content": content
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": assistant_text}
                ]
            }
        ]
    }


### Parallel Generation Pipeline

In [15]:
def _generate_one_record(record: dict) -> dict:
    """
    Worker untuk satu record.
    Mengembalikan raw_output saja; cleaning dan penulisan file dilakukan di main thread.
    """
    ext_path, int_path = get_image_paths_from_record(record)
    if ext_path is None and int_path is None:
        return {
            "status": "skip",
            "record": record,
            "raw_output": None,
            "reason": "no_image"
        }

    prompt_text = build_prompt_text(record)
    raw_output = call_openrouter(ext_path, int_path, prompt_text)

    return {
        "status": "ok",
        "record": record,
        "raw_output": raw_output,
        "reason": None,
    }


def generate_records_parallel(
    records: List[dict],
    output_path: Path,
    cache: Dict[str, dict],
    max_workers: int = MAX_WORKERS,
) -> List[dict]:
    """
    Generate label via OpenRouter secara paralel, lalu simpan sebagai JSONL SFT.

    Strategi:
    - record yang sudah ada di file output akan dilewati
    - record yang sudah ada di cache akan langsung dipakai tanpa API call
    - sisanya diproses paralel
    """
    output_samples_with_idx = []

    done_ids = set()
    if output_path.exists():
        with open(output_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                    if isinstance(obj, dict) and "id" in obj:
                        done_ids.add(obj["id"])
                except Exception:
                    pass

    print(f"\nGenerating {output_path.name} ...")
    print("Already done:", len(done_ids))

    # 1) pisahkan yang sudah ada di cache / file output dan yang perlu API call
    pending = []
    for idx, record in enumerate(records):
        key = sample_key(record)

        if key in done_ids:
            continue

        ext_path, int_path = get_image_paths_from_record(record)
        if ext_path is None and int_path is None:
            print(f"[SKIP] Tidak ada image valid: {record.get('house_id')}")
            continue

        if key in cache and cache[key].get("raw_output"):
            cleaned = clean_output(cache[key].get("raw_output"))
            if cleaned is not None:
                sample = build_sft_sample(record, cleaned)
                output_samples_with_idx.append((idx, sample))
                continue

        pending.append((idx, record, key))

    print("Cached samples:", len(output_samples_with_idx))
    print("Pending API calls:", len(pending))

    # 2) proses sisanya secara paralel
    if pending:
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_meta = {
                executor.submit(_generate_one_record, record): (idx, record, key)
                for idx, record, key in pending
            }

            for future in tqdm(as_completed(future_to_meta), total=len(future_to_meta), desc=output_path.stem):
                idx, record, key = future_to_meta[future]

                try:
                    result = future.result()
                except Exception as e:
                    print(f"[ERROR] Future gagal untuk {record.get('house_id')}: {e}")
                    continue

                if result["status"] != "ok":
                    print(f"[SKIP] {record.get('house_id')} | {result.get('reason')}")
                    continue

                raw_output = result["raw_output"]
                cleaned = clean_output(raw_output)

                # retry sekali jika output invalid
                if cleaned is None:
                    ext_path, int_path = get_image_paths_from_record(record)
                    prompt_text = build_prompt_text(record)
                    raw_output = call_openrouter(ext_path, int_path, prompt_text)
                    cleaned = clean_output(raw_output)

                cache[key] = {
                    "raw_output": raw_output,
                    "house_id": record.get("house_id"),
                    "split": record.get("split"),
                    "dataset_scheme": record.get("dataset_scheme"),
                }
                save_cache(CACHE_PATH, cache)

                if cleaned is None:
                    print(f"[DROP] Output invalid: {record.get('house_id')} | {record.get('dataset_scheme')}")
                    continue

                sample = build_sft_sample(record, cleaned)
                output_samples_with_idx.append((idx, sample))

    # 3) urutkan lalu tulis JSONL
    output_samples_with_idx = sorted(output_samples_with_idx, key=lambda x: x[0])
    output_samples = [sample for _, sample in output_samples_with_idx]

    with open(output_path, "a", encoding="utf-8") as fout:
        for sample in output_samples:
            fout.write(json.dumps(sample, ensure_ascii=False) + "\n")

    return output_samples


### Run Generation

In [18]:
cache = load_cache(CACHE_PATH)
print("Cache loaded:", len(cache))

train_records = records_by_split["train"]
val_records = records_by_split["val"]

# urutkan agar reproducible
train_records = sorted(train_records, key=lambda r: (r.get("dataset_scheme", ""), r.get("house_id", "")))
val_records   = sorted(val_records, key=lambda r: (r.get("dataset_scheme", ""), r.get("house_id", "")))

train_samples = generate_records_parallel(train_records, OUTPUT_TRAIN_JSONL, cache, max_workers=MAX_WORKERS)
val_samples   = generate_records_parallel(val_records, OUTPUT_VAL_JSONL, cache, max_workers=MAX_WORKERS)

# rebuild combined file
combined_samples = []
for path in [OUTPUT_TRAIN_JSONL, OUTPUT_VAL_JSONL]:
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    combined_samples.append(json.loads(line))

with open(OUTPUT_ALL_JSONL, "w", encoding="utf-8") as f:
    for sample in combined_samples:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

print("\nSelesai.")
print("Train samples:", len(train_samples))
print("Val samples:", len(val_samples))
print("Combined:", len(combined_samples))
print("Cache size:", len(cache))

Cache loaded: 15

Generating train.jsonl ...
Already done: 6
Cached samples: 0
Pending API calls: 3


train:  33%|███▎      | 1/3 [00:06<00:12,  6.12s/it]

[DROP] Output invalid: H00688 | single


train:  67%|██████▋   | 2/3 [00:08<00:03,  3.75s/it]

[DROP] Output invalid: H01185 | single


...

[DROP] Output invalid: H00736 | single

Generating val.jsonl ...
Already done: 4
Cached samples: 0
Pending API calls: 2


val:  50%|█████     | 1/2 [00:05<00:05,  5.84s/it]

[DROP] Output invalid: H00692 | single


val: 100%|██████████| 2/2 [00:08<00:00,  4.28s/it]

[DROP] Output invalid: H00516 | single

Selesai.
Train samples: 0
Val samples: 0
Combined: 10
Cache size: 15
